# 12d — IPCC Assessment-Report Model Grids

Summary statistics of climate-model grid resolution and vertical-level
counts, broken down by IPCC Assessment Report generation — FAR (1990)
through AR6 (2021), plus a hypothetical future "AR7" used illustratively.
The first four reports each analyze one assessment report's per-model grid
table (`chap12_ipcc_grid.xlsx`) and share near-identical structure, so
they're collapsed into a single parametrized loop here rather than four
repeated code blocks. The level-count and resolution summaries that follow
select one of several hardcoded literal arrays by assessment-report
number, using a plain Python dict keyed by AR number.


In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "../../data"


## Shared helpers

`resolution_to_angular` handles three different grid-resolution
conventions used across model generations — rhomboidal spectral
truncation (`"R15"`, `"R21"`, `"R30"`), triangular spectral truncation
(`"T21"` ... `"T319"`), or a plain angular degree string
(`"4°x5°"`, `"2.5°x3.75°"`) — converting each to a
$(\Delta\varphi,\Delta\lambda)$ angle pair before the horizontal-resolution
formula (introduced in the previous notebook) can be applied.
`compute_stats` reports sample size (including any `NaN`s), then
median/min/max/mean of the non-`NaN` values.


In [2]:
RHOMBOIDAL = {"R15": (4.5, 7.5), "R21": (3.2, 5.6), "R30": (2.25, 3.75)}
TRIANGULAR_PHI = {
    "T21": 5.6, "T30": 4.0, "T31": 3.9, "T32": 3.8, "T42": 2.8, "T47": 2.5,
    "T63": 1.9, "T85": 1.4, "T106": 1.1, "T126": 0.9375, "T159": 0.75,
    "T213": 0.5625, "T319": 0.375,
}


def resolution_to_angular(res):
    # Converts a rhomboidal, triangular, or plain angular-degree resolution
    # string to a (delta_phi, delta_lambda) angle pair.
    res = str(res)
    if res in RHOMBOIDAL:
        return RHOMBOIDAL[res]
    if res in TRIANGULAR_PHI:
        d = TRIANGULAR_PHI[res]
        return (d, d)
    if "x" in res:
        phi_str, lam_str = res.split("x")
        return (float(phi_str.replace("\u00b0", "")), float(lam_str.replace("\u00b0", "")))
    return (np.nan, np.nan)


def compute_horizontal_resolution_angle(delta_phi, delta_lambda):
    # Same formula as the shared compute_horizontal_resolution helper (12c), taking angles directly.
    length_phi = 111.2
    R_0 = length_phi * np.sqrt(delta_phi * delta_lambda * np.cos(0.0))
    R_45 = length_phi * np.sqrt(delta_phi * delta_lambda * np.cos(np.pi / 4))
    R_avg = 0.7628 * R_0
    return R_0, R_45, R_avg


def compute_stats(x):
    x = np.asarray(x, dtype=float)
    n = x.size
    y = x[~np.isnan(x)]
    return [n, np.median(y), np.min(y), np.max(y), np.mean(y)]


STATS_COLS = ["n", "median", "min", "max", "mean"]


## 1-4. Per-assessment-report grid-resolution and level-count summaries

For each of FAR, SAR, TAR, and AR4: the atmosphere-model grid resolution
$R_0$ (equator), $R_{45}$ (45°), $R_{avg}$ (the `0.7628` shortcut
introduced in the previous notebook), and $R_{typical}=\sqrt{R_0\cdot R_{45}}$,
each summarized across all models in the table, followed by the
vertical-level-count summary (`A_Level` atmosphere, `O_Level` ocean where
available). FAR's table has no ocean-grid-resolution column
(`O_Resolution` is all missing — ocean components weren't consistently
gridded/reported this early), so only FAR skips the ocean-resolution block
that SAR/TAR/AR4 all have.


In [3]:
def analyze_ar_sheet(sheet, has_ocean):
    T = pd.read_excel(f"{DATA_DIR}/chap12_ipcc_grid.xlsx", sheet_name=sheet)
    T["A_angle_phi"], T["A_angle_lambda"] = zip(*T["A_Resolution"].map(resolution_to_angular))

    def resolution_stats_block(delta_phi, delta_lambda):
        R_0, R_45, R_avg = compute_horizontal_resolution_angle(delta_phi.values, delta_lambda.values)
        R_typical = np.sqrt(R_0 * R_45)
        rows = {
            "R_0 (phi=0)": compute_stats(R_0),
            "R_45 (phi=45deg)": compute_stats(R_45),
            "R_avg": compute_stats(R_avg),
            "R_typical": compute_stats(R_typical),
        }
        return pd.DataFrame(rows, index=STATS_COLS).T

    print(f"=== {sheet}: atmosphere grid resolution (km) ===")
    display(resolution_stats_block(T["A_angle_phi"], T["A_angle_lambda"]).round(0))

    print(f"=== {sheet}: vertical-level counts ===")
    level_rows = {"A_Level (atmosphere)": compute_stats(T["A_Level"])}
    if has_ocean:
        level_rows["O_Level (ocean)"] = compute_stats(T["O_Level"])
    display(pd.DataFrame(level_rows, index=STATS_COLS).T.round(1))

    if has_ocean:
        T["O_angle_phi"], T["O_angle_lambda"] = zip(*T["O_Resolution"].map(resolution_to_angular))
        print(f"=== {sheet}: ocean grid resolution (km) ===")
        display(resolution_stats_block(T["O_angle_phi"], T["O_angle_lambda"]).round(0))
    print()


for sheet, has_ocean in [("FAR", False), ("SAR", True), ("TAR", True), ("AR4", True)]:
    analyze_ar_sheet(sheet, has_ocean)


=== FAR: atmosphere grid resolution (km) ===


,n,median,min,max,mean
R_0 (phi=0),26.0,646.0,323.0,995.0,634.0
R_45 (phi=45deg),26.0,543.0,272.0,836.0,533.0
R_avg,26.0,493.0,246.0,759.0,483.0
R_typical,26.0,592.0,296.0,912.0,581.0


=== FAR: vertical-level counts ===


,n,median,min,max,mean
A_Level (atmosphere),26.0,9.0,2.0,19.0,9.3



=== SAR: atmosphere grid resolution (km) ===


...

,n,median,min,max,mean
R_0 (phi=0),23.0,278.0,122.0,497.0,293.0
R_45 (phi=45deg),23.0,234.0,103.0,418.0,247.0
R_avg,23.0,212.0,93.0,379.0,224.0
R_typical,23.0,255.0,112.0,456.0,269.0


=== AR4: vertical-level counts ===


,n,median,min,max,mean
A_Level (atmosphere),23.0,24.0,12.0,56.0,26.2
O_Level (ocean),23.0,31.0,13.0,47.0,29.7


=== AR4: ocean grid resolution (km) ===


,n,median,min,max,mean
R_0 (phi=0),23.0,139.0,27.0,497.0,175.0
R_45 (phi=45deg),23.0,117.0,23.0,418.0,147.0
R_avg,23.0,106.0,21.0,379.0,134.0
R_typical,23.0,127.0,25.0,456.0,161.0


## 5. Vertical-level counts across all 7 report generations

Extends the FAR-AR4 level-count summaries above with hardcoded per-model
level-count literals for all 7 assessment-report generations (FAR=AR1
through AR6, plus an illustrative future "AR7"), separately for
atmosphere (AGCM) and ocean (OGCM) components. The FAR/SAR/TAR/AR4
literals here reproduce the same `A_Level`/`O_Level` columns used above
(confirmed matching on manual inspection) — AR5/AR6/AR7 have no grid table
in `chap12_ipcc_grid.xlsx` at all, so these literals are the only source
for them.

Selecting one of the 7 array pairs by assessment-report number uses a
plain Python dict keyed by AR number.


In [4]:
L_LEVELS = {
    1: ([9,9,2,2,5,9,9,9,4,7,9,9,9,9,11,11,11,11,11,10,9,11,9,9,19,19],
        [12,4,11,9]),
    2: ([9,10,31,9,9,14,9,9,2,15,19,19,15,9,9,19],
        [12,29,20,16,12,18,13,16,20,20,11,9,21,20,15,20]),
    3: ([30,19,9,17,20,10,10,9,18,9,18,18,19,19,19,9,9,14,9,9,9,19,19,15,15,15,30,9,18,18,20],
        [31,31,12,12,17,29,29,20,20,21,45,45,11,11,11,12,12,18,16,13,20,20,20,31,31,21,23,20,25,32,17]),
    4: ([16,31,26,32,31,45,18,31,19,26,24,24,12,20,20,21,19,56,20,30,26,19,38],
        [30,35,40,29,29,31,31,40,20,16,np.nan,np.nan,16,16,13,33,31,47,43,23,40,20,40]),
    5: ([38,38,26,26,26,35,35,27,27,27,30,66,27,39,31,95,31,18,62,26,26,26,24,48,24,24,32,
         32,40,40,40,40,19,60,60,38,21,39,39,39,56,40,80,80,47,95,47,64,64,48,48,64,26,26],
        [50,50,40,40,50,40,40,60,60,60,60,60,60,31,31,31,42,31,31,30,30,40,50,50,63,50,
         np.nan,26,26,32,32,20,np.nan,np.nan,40,40,31,31,31,48,50,44,44,40,40,40,np.nan,np.nan,51,51,40,53,53]),
    6: ([30,47,46,26,31,32,26,49,64,30,91,91,38,85,66,91,91,26,47,21,73,79,79,32,
         40,81,85,85,95,80,40,32,26,32,85,33,49,47,30,30,14],
        [60,46,40,40,50,30,30,45,50,50,75,75,50,50,60,75,75,60,40,40,40,75,30,52,
         63,63,75,75,40,61,40,60,53,70,50,75,75,46,60,46,18]),
    7: ([95,56,32,26,26,91,91,91,91,73,85,85,85,95,95,30,33],
        [46,40,55,50,50,75,75,75,75,40,75,75,75,40,40,62,75]),
}


def grid_analyze_levels(ar):
    agcm, ogcm = L_LEVELS[ar]
    return compute_stats(agcm), compute_stats(ogcm)


rows_agcm, rows_ogcm = {}, {}
for ar in range(1, 8):
    stat_agcm, stat_ogcm = grid_analyze_levels(ar)
    rows_agcm[f"AR{ar}"] = stat_agcm
    rows_ogcm[f"AR{ar}"] = stat_ogcm

print("=== Vertical-level counts, AGCM (atmosphere) ===")
display(pd.DataFrame(rows_agcm, index=STATS_COLS).T.round(1))
print("=== Vertical-level counts, OGCM (ocean) ===")
display(pd.DataFrame(rows_ogcm, index=STATS_COLS).T.round(1))


=== Vertical-level counts, AGCM (atmosphere) ===


,n,median,min,max,mean
AR1,26.0,9.0,2.0,19.0,9.3
AR2,16.0,9.5,2.0,31.0,12.9
AR3,31.0,17.0,9.0,30.0,15.5
AR4,23.0,24.0,12.0,56.0,26.3
AR5,54.0,38.0,18.0,95.0,41.2
AR6,41.0,47.0,14.0,95.0,53.0
AR7,17.0,85.0,26.0,95.0,69.4


=== Vertical-level counts, OGCM (ocean) ===


,n,median,min,max,mean
AR1,4.0,10.0,4.0,12.0,9.0
AR2,16.0,17.0,9.0,29.0,17.0
AR3,31.0,20.0,11.0,45.0,21.8
AR4,23.0,31.0,13.0,47.0,29.7
AR5,53.0,40.0,20.0,63.0,42.6
AR6,41.0,50.0,18.0,75.0,53.9
AR7,17.0,62.0,40.0,75.0,60.2


## 6. Horizontal resolution (km) for AR6/AR7

The same summary approach as above, but for hardcoded *horizontal
resolution in km* literals rather than vertical-level counts — the
magnitudes here, tens to hundreds of km, match the $R_0$/$R_{45}$-scale
resolution values from the sections above rather than a level-count
range. Only AR6 and AR7 are covered, since the earlier generations'
resolution-in-km already came out of the angle-based formula above.


In [5]:
L_RESOLUTION_KM = {
    6: ([100,170,80,100,250,100,90,190,250,170,100,140,50,140,140,140,100,80,120,80,
         100,170,150,150,160,240,190,250,120,140,60,140,170,80,100,200,200,100,190,190,190,
         100,140,100,100,170,100,100,260],
        [60,50,20,80,80,90,80,80,70,90,70,70,20,70,70,70,40,70,70,70,
         60,100,70,30,70,150,90,80,80,70,20,70,100,40,60,100,70,60,60,60,60,60,90,
         20,40,70,60,60,190]),
    7: ([80,40,20,100,20,50,40,60,30,50,60,30,30,80,40,30,50],
        [20,20,8,20,20,20,20,20,20,10,7,20,7,40,40,8,20]),
}

rows_agcm2, rows_ogcm2 = {}, {}
for ar in (6, 7):
    agcm, ogcm = L_RESOLUTION_KM[ar]
    rows_agcm2[f"AR{ar}"] = compute_stats(agcm)
    rows_ogcm2[f"AR{ar}"] = compute_stats(ogcm)

print("=== Horizontal resolution (km), AGCM (atmosphere) ===")
display(pd.DataFrame(rows_agcm2, index=STATS_COLS).T.round(1))
print("=== Horizontal resolution (km), OGCM (ocean) ===")
display(pd.DataFrame(rows_ogcm2, index=STATS_COLS).T.round(1))


=== Horizontal resolution (km), AGCM (atmosphere) ===


,n,median,min,max,mean
AR6,49.0,140.0,50.0,260.0,141.8
AR7,17.0,40.0,20.0,100.0,47.6


=== Horizontal resolution (km), OGCM (ocean) ===


,n,median,min,max,mean
AR6,49.0,70.0,20.0,190.0,69.6
AR7,17.0,20.0,7.0,40.0,18.8
